# Evaluate scalars

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import glob
import re
from typing import Dict, List, Optional, Tuple
from collections import defaultdict


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import sem

import torch

from urbanmarl.eval_results import (
    find_experiment_dirs,
    get_scalars_dir,
    parse_metric_from_filename,
    load_csv_metric,
    load_all_metrics,
    aggregate_runs,
    plot_metric,
    #
    aggregate_runs_by_envs,
    plot_metric_by_envs,
    #
    plot_all_metrics
    )
import math

In [ ]:
# test_los: has the following conf
# max_horizontal_speed: 490
# max_vertical_speed: 12.0
exp_dir = "navigate_1"

# test_los: has the following conf
# max_horizontal_speed: 500.0
# max_vertical_speed: 50.0
# output_dir = project_root / "outputs" / "test_los_velocity"
# exp_dir = "test_los_velocity"



experiments_dir = project_root / "outputs" / exp_dir
output_dir = project_root / "outputs" / "plots" / exp_dir

experiments = find_experiment_dirs(experiments_dir)
if not experiments:
    print(f"No experiment directories found in {experiments_dir}")

print(f"Found {len(experiments)} experiment directories.")

In [ ]:
colliction_info_metrics = [
    'collection_info_collisions',
    'collection_info_los',
    'collection_info_velocity',
]
info_metrics = [
    'reward',
    'los',
    'collisions',
    'velocity',
]
timer_metrics = [
    'timers_collection_time',
    'timers_evaluation_time',
    'timers_iteration_time',
    'timers_total_time',
    'timers_training_time',
]
report_metrics = [
    'collection_info_collisions',
    'collection_info_los',
    'collection_info_velocity',
    #
    'eval_reward_episode_len_mean',
    #
    'timers_collection_time',
    'timers_evaluation_time',
    'timers_iteration_time',
    'timers_total_time',
    'timers_training_time',
    #
    'train_uav_ESS',
    'train_uav_alpha',
    'train_uav_clip_fraction',
    'train_uav_entropy',
    'train_uav_explained_variance',
    'train_uav_grad_norm_loss_actor',
    'train_uav_grad_norm_loss_alpha',
    'train_uav_grad_norm_loss_critic',
    'train_uav_grad_norm_loss_objective',
    'train_uav_grad_norm_loss_qvalue',
    'train_uav_grad_norm_loss_value',
    'train_uav_kl_approx',
    'train_uav_loss_actor',
    'train_uav_loss_alpha',
    'train_uav_loss_critic',
    'train_uav_loss_entropy',
    'train_uav_loss_objective',
    'train_uav_loss_qvalue',
    'train_uav_loss_value',
    'train_uav_pred_value',
    'train_uav_pred_value_max',
    'train_uav_target_value',
    'train_uav_target_value_max',
    'train_uav_td_error',
    #
    'collisions',
    'los',
    'reward',
    'velocity'
]

In [ ]:
def metric_labels(
    metric_name,
    info_metrics = ['reward', 'los', 'collisions', 'velocity']
):
    if metric_name in info_metrics:
        x_label = 'E'
        title = f"{metric_name} per Urban Environements".title()
    else:
        x_label = 'Episode'
        title = f"{metric_name}".replace('_', ' ').title()
    if metric_name.startswith('timers_'):
        y_label = f"{metric_name.replace('timers_', '_')}".replace('_', ' ').title() + ' (s)'
    elif metric_name.startswith('train_'):
        y_label = metric_name.replace('train_', '').replace('_', ' ').title()
    elif metric_name == 'eval_reward_episode_len_mean':
        y_label = 'Mean Steps per Episode'.title()
    else:
        y_label = metric_name.replace('_', ' ').title().replace('Los', 'LoS')
    return x_label, y_label, title

metric_name = 'timers_collection_time'
metric_labels(metric_name)      

In [ ]:
all_data = load_all_metrics(experiments)



In [ ]:
keys = list(all_data.keys())
sorted(keys)

In [ ]:

png = output_dir / 'png'
plot_all_metrics(all_data, png)

In [ ]:
def calculate_percentile(group, percentile):
    return np.percentile(group, percentile)

percentile_95 = lambda x: calculate_percentile(x, 95)

def plot_group_metrics(
    dictionary: Dict,
    output_dir, 
    metric_list= [],
    n_cols = 2,
    projection = 'cartesian',
    file_name = 'compare_metrics',
    figsize = (7, 6),
    pdf=False,
    fill = True
):
    n_metrics = len(metric_list)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    if projection.lower() == 'polar':
        fig, axes = plt.subplots(n_rows, n_cols, 
                                 subplot_kw={'projection': 'polar'},
                                 figsize=(figsize[0]*n_cols, figsize[1]*n_rows+1),
                                 squeeze=False)
    else:
        fig, axes = plt.subplots(n_rows, n_cols, 
                                 # subplot_kw={'projection': 'polar'},
                                 figsize=(figsize[0]*n_cols, figsize[1]*n_rows),
                                 squeeze=False)
    
    for idx, metric in enumerate(metric_list):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        #
        con_df = pd.concat(dictionary[metric])
        #
        for algo in con_df.algo.unique():
            _df = con_df.loc[con_df['algo'] == algo]
            if 'env_id' in _df.columns:
                # 
                x_label = 'env_id'
                df = _df.groupby([x_label])[['value']].agg(percentile_95)
                # df = _df.groupby(['env_id'])[['value']].agg(['max'])
                df.reset_index(inplace=True)
                df.columns = [x_label, 'mean']
                df['std'] = df['mean'].std()
                E = df[x_label]
                metric_mean = df['mean'] 
                metric_std = df['std'] 
                # 
                ax.plot(E, metric_mean, label= algo.upper(), linewidth=2.0)
                if fill:
                    # min_fill = np.clip(metric_mean - metric_std, a_min=np.min(metric_mean), a_max=None)
                    # max_fill = np.clip(metric_mean + metric_std, a_min=None, a_max=np.max(metric_mean))
                    # ax.fill_between(E, min_fill, max_fill, alpha=0.2)
                    ax.fill_between(E, metric_mean - metric_std, metric_mean + metric_std, alpha=0.2)
            else:
                x_label = 'step'
                df = _df.groupby([x_label])[['value']].agg(percentile_95)
                # df = df.groupby(['env_id'])[['value']].agg(['max'])
                df.reset_index(inplace=True)
                df.columns = [x_label, 'mean']
                df['std'] = df['mean'].std()
                E = df[x_label]
                metric_mean = df['mean'] 
                metric_std = df['std'] 
                #
                ax.plot(E, metric_mean, label= algo.upper(), linewidth=2.0)
                if fill:
                    # min_fill = np.clip(metric_mean - metric_std, a_min=np.min(metric_mean), a_max=None)
                    # max_fill = np.clip(metric_mean + metric_std, a_min=None, a_max=np.max(metric_mean))
                    # ax.fill_between(E, min_fill, max_fill, alpha=0.2)
                    ax.fill_between(E, metric_mean - metric_std, metric_mean + metric_std, alpha=0.2)
            
        #
        x_label, y_label, title = metric_labels(metric)
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        ax.legend()
    #
    # plt.legend()
    # plt.grid(True, alpha=0.3)
    # 
    # Save
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        if pdf:
            save_path = os.path.join(output_dir, f'{file_name}.pdf')
        else:
            save_path = os.path.join(output_dir, f'{file_name}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved plot: {save_path}")
    else:
        plt.show()
    plt.close()

def plot_catalogue(
    dictionary: Dict,
    output_dir, 
    figsize = (7, 6),
    pdf=False,
    fill = True
):
    catalog_dir = output_dir / 'catalogue'
    # draw all metrics
    plot_all_metrics(dictionary, catalog_dir, pdf=pdf)
    # plot groups
    time_metrics = [
        'timers_collection_time',
        'timers_training_time',
        'timers_evaluation_time',
        'timers_iteration_time',
    ]
    plot_group_metrics(
        dictionary,
        catalog_dir, 
        metric_list= time_metrics,
        n_cols = 2,
        projection = 'cartesian',
        file_name = 'grouped_timers',
        figsize = figsize,
        pdf=pdf
    )
    #
    info_metrics = [
        'los',
        'collisions',
        'velocity',
        'reward',
    ]
    plot_group_metrics(
        dictionary,
        catalog_dir, 
        metric_list= info_metrics,
        n_cols = 2,
        projection = 'polar',
        file_name = 'grouped_info_metrics',
        figsize = figsize,
        pdf=pdf
    )

def plot_report(
    dictionary: Dict,
    output_dir, 
    figsize = (7, 6),
    pdf=True,
    fill = True
):
    # draw all metrics
    info_metrics = [
        'los',
        'collisions',
        'velocity',
        'reward',
    ]
    new_dict = {metric: dfs for metric, dfs in dictionary.items() if metric in info_metrics}
    plot_all_metrics(new_dict, output_dir, pdf=pdf)
    # plot groups
    time_metrics = [
        'timers_collection_time',
        'timers_training_time',
        'timers_evaluation_time',
        'timers_iteration_time',
    ]
    plot_group_metrics(
        dictionary,
        output_dir, 
        metric_list= time_metrics,
        n_cols = 2,
        projection = 'cartesian',
        file_name = 'grouped_timers',
        figsize = figsize,
        pdf=pdf
    )
    #
    info_metrics = [
        'los',
        'collisions',
        'velocity',
        'reward',
    ]
    plot_group_metrics(
        dictionary,
        output_dir, 
        metric_list= info_metrics,
        n_cols = 2,
        projection = 'polar',
        file_name = 'grouped_info_metrics',
        figsize = figsize,
        pdf=pdf
    )

def plot_training_metrics(
    dictionary: Dict,
    output_dir, 
    figsize = (7, 6),
    n_cols = 3,
    pdf=True,
    fill = False
):
    training_metrics = [
        'train_uav_ESS',
        'train_uav_alpha',
        'train_uav_clip_fraction',
        'train_uav_entropy',
        'train_uav_explained_variance',
        'train_uav_grad_norm_loss_actor',
        'train_uav_grad_norm_loss_alpha',
        'train_uav_grad_norm_loss_critic',
        'train_uav_grad_norm_loss_objective',
        'train_uav_grad_norm_loss_qvalue',
        'train_uav_grad_norm_loss_value',
        'train_uav_kl_approx',
        'train_uav_loss_actor',
        'train_uav_loss_alpha',
        'train_uav_loss_critic',
        'train_uav_loss_entropy',
        'train_uav_loss_objective',
        'train_uav_loss_qvalue',
        'train_uav_loss_value',
        'train_uav_pred_value',
        'train_uav_pred_value_max',
        'train_uav_target_value',
        'train_uav_target_value_max',
        'train_uav_td_error',
    ]
    plot_group_metrics(
        dictionary,
        output_dir, 
        metric_list= training_metrics,
        n_cols = n_cols,
        projection = 'cartesian',
        file_name = 'grouped_training_metrics',
        figsize = figsize,
        pdf=pdf,
        fill = fill
    )

In [ ]:
plot_training_metrics(
    all_data,
    output_dir, 
    figsize = (7, 6),
    n_cols = 4,
    pdf=True,
    fill = False
)

In [ ]:
plot_report(
    all_data,
    output_dir, 
    figsize = (7, 6),
    pdf=True
)

In [ ]:


plot_catalogue(
    all_data,
    output_dir, 
    figsize = (7, 6),
    pdf=False
)

In [ ]:
##
figure_size = (7, 6)
time_metrics = [
    'timers_collection_time',
    'timers_training_time',
    'timers_evaluation_time',
    'timers_iteration_time',
    # 'timers_total_time',
]

png = output_dir / 'png'

plot_group_metrics(
    all_data,
    png, 
    metric_list= time_metrics,
    n_cols = 2,
    # projection = 'cartesian',
    file_name = 'grouped_timers',
    figsize = figure_size,
    pdf=False
)

##
info_metrics = [
    'los',
    'collisions',
    'velocity',
    'reward',
]

png = output_dir / 'png'

plot_group_metrics(
    all_data,
    png, 
    metric_list= info_metrics,
    n_cols = 2,
    projection = 'polar',
    file_name = 'grouped_info_metrics',
    figsize = figure_size,
    pdf=False
)

In [ ]:
png = output_dir / 'png'
plot_all_metrics(all_data, png)

In [ ]:
metrics_of_interest = [
    # 'collection_agents_reward_episode_reward_mean',
    # 'eval_agents_reward_episode_reward_mean',
    # 'collection_reward_episode_reward_mean',
    # 'eval_reward_episode_reward_mean',
    
    # Add any other metrics you want to plot by default
    'reward',
    'velocity',
    'los',
    'collisions',
]

# all_data = load_all_metrics(experiments, metrics_of_interest)
all_data = load_all_metrics(experiments)
print(f"Loaded data for metrics: {list(all_data.keys())}")

In [ ]:
def calculate_percentile(group, percentile):
    return np.percentile(group, percentile)

percentile_95 = lambda x: calculate_percentile(x, 95)


def plot_all_metrics(
    dictionary: Dict,
    output_dir, 
    algo_names=None,
    pdf=False
):
    for metric, value in dictionary.items():
        con_df = pd.concat(all_data[metric])
        #
        if 'env_id' in con_df.columns:
            fig, ax = plt.subplots(subplot_kw=dict(projection="polar"), figsize=(8, 6))  
        else:
            fig, ax = plt.subplots(figsize=(8, 6))  
        for algo in con_df.algo.unique():
            _df = con_df.loc[con_df['algo'] == algo]
            if 'env_id' in _df.columns:
                # 
                x_label = 'env_id'
                df = _df.groupby([x_label])[['value']].agg(percentile_95)
                # df = _df.groupby(['env_id'])[['value']].agg(['max'])
                df.reset_index(inplace=True)
                df.columns = [x_label, 'mean']
                df['std'] = df['mean'].std()
                E = df[x_label]
                metric_mean = df['mean'] 
                metric_std = df['std'] 
                # 
                ax.plot(E, metric_mean, label= algo.upper())
                ax.fill_between(E, metric_mean - metric_std, metric_mean + metric_std, alpha=0.2)
            else:
                x_label = 'step'
                df = _df.groupby([x_label])[['value']].agg(percentile_95)
                # df = df.groupby(['env_id'])[['value']].agg(['max'])
                df.reset_index(inplace=True)
                df.columns = [x_label, 'mean']
                df['std'] = df['mean'].std()
                E = df[x_label]
                metric_mean = df['mean'] 
                metric_std = df['std'] 
                #
                ax.plot(E, metric_mean, label= algo.upper())
                ax.fill_between(E, metric_mean - metric_std, metric_mean + metric_std, alpha=0.2)
            
        # ax.set(xlabel=x_label.title(), ylabel=f"{metric.title()}",
        #        title=f"{metric} over Urban Environements".title() if x_label=='env_id' else f"{metric}".upper()
        #       )
        #
        if x_label == 'env_id':
            # tick_vals = [-math.pi, -math.pi/2, 0, math.pi/2, math.pi]
            # tick_labels = ['-π', '-π/2', '0', 'π/2', 'π']
            # ax.set_xticks(tick_vals)
            # ax.set_xticklabels(tick_labels)
            plt.xlabel('E')
            
        else:
            plt.xlabel('Iteration')
        plt.ylabel(metric.upper())
        plt.title(f"{metric} over Urban Environements".title() if x_label=='env_id' else f"{metric}".upper())
        plt.legend()
        plt.grid(True, alpha=0.3)
        # 
        # Save
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            if pdf:
                save_path = os.path.join(output_dir, f'{metric}.pdf')
            else:
                save_path = os.path.join(output_dir, f'{metric}.png')
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"Saved plot: {save_path}")
        else:
            plt.show()
        plt.close()

png = output_dir / 'png'
plot_all_metrics(all_data, png)

In [ ]:
type(all_data['velocity'][0])
len(all_data['velocity'])

env_algo_group = defaultdict(list)
collect_algo_group = defaultdict(list)

df_list = all_data['velocity']
result = pd.DataFrame(columns=['algorithm', 'env_id', 'mean', 'max', 'min', 'std', 'median']) 
for df in df_list:
    algo = df['algo'].iloc[0]  # all rows have same algo
    # For per-env rewards, we might want to further split by env_id
    if 'env_id' in df.columns:
        value = df['value']
        d = {
            # 'algorithm': algo,
            'env_id': df['env_id'].iloc[0],
            'mean': value.mean(),
            'max': value.max(),
            'min': value.min(),
            'std': value.std(),
            'median': value.median()
        }
        # env_id = df['env_id'].iloc[0]
        env_algo_group[algo].append(d)
    else:
        collect_algo_group[algo].append(df)

In [ ]:
aggregated_all = {}

for metric_name, dfs_list in all_data.items():
    # Group DataFrames by algorithm
    env_algo_group = defaultdict(list)
    collect_algo_group = defaultdict(list)
    # algo_groups = defaultdict(list)
    for df in dfs_list:
        algo = df['algo'].iloc[0]  # all rows have same algo
        # For per-env rewards, we might want to further split by env_id
        if 'env_id' in df.columns:
            env_id = df['env_id'].iloc[0]
            env_algo_group[algo]
        else:
            collect_algo_group[algo].append(df)

    # aggregate env_algo_group
    aggregated_envs = {}
    for algo_name, env_dfs in env_algo_group.items():
        
    # Aggregate each group
    aggregated_for_metric = {}
    for group_key, group_dfs in algo_groups.items():
        # print(group_key, group_dfs)
        agg_df = aggregate_runs_by_envs(group_dfs)
        if agg_df is not None:
            aggregated_for_metric[group_key] = agg_df

    if aggregated_for_metric:
        aggregated_all[metric_name] = aggregated_for_metric

        # Plot this metric
        plot_metric_by_envs(metric_name, aggregated_for_metric, output_dir, pdf=True)

In [ ]:


aggregated_all = {}

for metric_name, dfs_list in all_data.items():
    # Group DataFrames by algorithm
    algo_groups = defaultdict(list)
    for df in dfs_list:
        algo = df['algo'].iloc[0]  # all rows have same algo
        # For per-env rewards, we might want to further split by env_id
        if 'env_id' in df.columns:
            # For simplicity, we treat each (algo, env_id) as separate group
            # but we can also aggregate across envs. We'll plot env-specific lines.
            # We'll create a key: algo + '_env' + env_id
            env_id = df['env_id'].iloc[0]
            # group_key = f"{algo}_env{env_id}"
        #     algo_groups[algo].append(df)
        # else:
        #     algo_groups[algo].append(df)

        # all
        algo_groups[algo].append(df)
        # last 5 
        # if len(df) >= 5:
        #     df_end = df.iloc[-5:]
        # else:
        #     df_end = df
        # algo_groups[algo].append(df_end)

    # Aggregate each group
    aggregated_for_metric = {}
    for group_key, group_dfs in algo_groups.items():
        # print(group_key, group_dfs)
        agg_df = aggregate_runs_by_envs(group_dfs)
        if agg_df is not None:
            aggregated_for_metric[group_key] = agg_df

    if aggregated_for_metric:
        aggregated_all[metric_name] = aggregated_for_metric

        # Plot this metric
        plot_metric_by_envs(metric_name, aggregated_for_metric, output_dir, pdf=True)

In [ ]:
aggregated_all.keys()

In [ ]:
for metric in metrics_of_interest:
    fig, ax = plt.subplots(subplot_kw=dict(projection="polar"))
    
    for algo, df in aggregated_all[metric].items():
        E = df['E']
        metric_mean = df['mean'] 
        metric_std = df['std'] 
        ax.plot(E, metric_mean, label= algo.upper())
        # ax.fill_between(E, metric_mean - metric_std, metric_mean + metric_std, alpha=0.2)
        
    ax.set(xlabel='E', ylabel=f"{metric.title()}",
           title=f"{metric} over Urban Environements".title())
    # ax.grid()
    plt.legend()
    # fig.savefig("test.png")
    plt.show()